# L04 – Data in i modellen: Dataset, DataLoader och normalisering

**Tid:** ca 1,5 h · **Del A: Grunder**

## Mål
1. Förstå vad `Dataset` och `DataLoader` gör och skriva ett eget `Dataset`.
2. Ladda bilddata (Fashion-MNIST) och dela upp den i train/val/test.
3. Normalisera med statistik **bara från train**, som `StandardScaler` i en `Pipeline`.
4. Förstå **varför** normalisering gör träningen snabbare och stabilare.

**Var i arbetsflödet?** Steg 2 och 4: uppdelning och datapipeline.

In [ ]:
import random
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms

def set_seed(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "../data"   # dl-course/data, ignoreras av git
print("device:", device)

## 1. Dataset och DataLoader – arbetsfördelningen

| Klass | Ansvar | Fråga den svarar på |
|---|---|---|
| `Dataset` | att hålla datan och hämta **ett** exempel | "Hur många exempel finns det, och vad är exempel nummer *i*?" |
| `DataLoader` | att sätta ihop exempel till **batcher**, blanda dem och ladda dem parallellt | "Ge mig nästa batch." |

I L03 gjorde vi batchningen för hand med `randperm`. `DataLoader` gör samma sak, men den fungerar också när datan inte får plats i minnet. Bilder läses då från disk en i taget.

Ett eget `Dataset` behöver bara två metoder:

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):                 # hur många exempel?
        return len(self.X)

    def __getitem__(self, idx):        # exempel nummer idx
        return self.X[idx], self.y[idx]

ds = TabularDataset(np.random.rand(10, 3), np.arange(10) % 2)
print("len:", len(ds))
print("exempel 0:", ds[0])

loader = DataLoader(ds, batch_size=4, shuffle=True)
for X_batch, y_batch in loader:
    print("batch:", X_batch.shape, y_batch.tolist())

Observera att den sista batchen bara har 2 exempel (10 = 4 + 4 + 2). Det är normalt. Med `drop_last=True` hoppas den över, men det behövs sällan.

## 2. Fashion-MNIST

Fashion-MNIST består av 70 000 gråskalebilder (28×28) av klädesplagg i 10 klasser: 60 000 för träning och 10 000 för test. Datasetet är svårare än siffrorna i MNIST men fortfarande snabbt att träna på. Vi använder det i flera lektioner.

`torchvision.datasets` laddar ned datan första gången. `transforms.ToTensor()` gör om varje bild från PIL-format (pixlar 0–255) till en tensor med shape `(1, 28, 28)` och värden i `[0, 1]`.

In [ ]:
raw_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transforms.ToTensor())
raw_test = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transforms.ToTensor())
classes = raw_train.classes
print(len(raw_train), "train,", len(raw_test), "test")
print("klasser:", classes)

img, label = raw_train[0]
print("en bild:", img.shape, img.dtype, f"min={img.min():.2f} max={img.max():.2f}", "etikett:", label)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, i in zip(axes.flat, range(16)):
    img, label = raw_train[i]
    ax.imshow(img.squeeze(), cmap="gray")   # squeeze: (1, 28, 28) → (28, 28)
    ax.set_title(classes[label], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Train, val och test

Fashion-MNIST har redan en färdig testdel. Vi delar de 60 000 träningsbilderna i **50 000 train** och **10 000 val** med `random_split`. Samma seed (via `generator`) ger samma uppdelning varje gång. Det är viktigt när du jämför experiment.

In [ ]:
generator = torch.Generator().manual_seed(42)
train_part, val_part = random_split(raw_train, [50_000, 10_000], generator=generator)
print("train:", len(train_part), " val:", len(val_part), " test:", len(raw_test))
print("första index i train:", train_part.indices[:5])

## 4. Normalisering

### Varför normalisera?
Ett neuralt nät tränas med gradient descent. Om indata ligger runt 0 med standardavvikelse cirka 1 händer följande:
- Gradienterna får en rimlig storlek, och **samma learning rate** fungerar bra för alla vikter.
- Aktiveringsfunktionerna (L05) arbetar i sitt "känsliga" område.
- Standardinitieringen av vikterna i PyTorch förutsätter ungefär den skalan.

Det är samma anledning som till att `StandardScaler` behövs för `LogisticRegression` och SVM i sklearn.

### Beräkna statistiken på train – och bara där
`datasets.FashionMNIST` har alla pixlar i `.data` (uint8, 0–255). Vi plockar ut **bara träningsindexen**:

In [ ]:
train_pixels = raw_train.data[train_part.indices].float() / 255.0   # (50000, 28, 28) i [0, 1]
MEAN = train_pixels.mean().item()
STD = train_pixels.std().item()
print(f"mean={MEAN:.4f}  std={STD:.4f}")

För bilder normaliserar vi med **ett** medelvärde och **en** standardavvikelse per *kanal*, inte per pixel. Gråskalebilder har en kanal och färgbilder tre (RGB, L10). Skälet är att pixlarna mäter samma sak (ljusstyrka), och vi vill inte förstöra att mittpixeln och hörnpixeln är jämförbara.

`transforms.Compose` sätter ihop steg som körs på varje bild när den hämtas, ungefär som en sklearn-`Pipeline`:

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                      # PIL 0–255 → tensor 0–1
    transforms.Normalize((MEAN,), (STD,)),      # (x - mean) / std
])

full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transform)
test_ds = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transform)
generator = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full_train, [50_000, 10_000], generator=generator)

x, _ = train_ds[0]
print(f"efter normalisering: mean={x.mean():.2f} std={x.std():.2f} min={x.min():.2f} max={x.max():.2f}")

## 5. DataLoaders

| Parameter | Train | Val/test | Varför |
|---|---|---|---|
| `batch_size` | 128 | 128 (eller större) | 32–256 är vanligt (L03). Vid utvärdering behövs inga gradienter, så större batcher får plats. |
| `shuffle` | **True** | **False** | Blanda för träningens skull. Vid utvärdering spelar ordningen ingen roll, och fast ordning gör resultaten jämförbara. |
| `num_workers` | 2 | 2 | Hjälpprocesser som förbereder nästa batch medan GPU:n räknar. 0 betyder allt i huvudprocessen, vilket är enklast att felsöka men långsammare. |

In [ ]:
BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

X_batch, y_batch = next(iter(train_loader))
print("X_batch:", X_batch.shape, X_batch.dtype)   # (128, 1, 28, 28)
print("y_batch:", y_batch.shape, y_batch.dtype)   # (128,) int64, som CrossEntropy vill ha
print("antal batcher per epok:", len(train_loader))

Så här ser en träningsepok ut med en `DataLoader`. Det är samma loop som i L03, men nu kommer batcherna från loadern:

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    for X, y in loader:
        X, y = X.to(device), y.to(device)   # flytta varje batch till GPU
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()

@torch.no_grad()   # som "with torch.no_grad()" runt hela funktionen
def accuracy(model, loader):
    model.eval()
    correct = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        correct += (model(X).argmax(dim=1) == y).sum().item()
    return correct / len(loader.dataset)

Som modell använder vi **multinomial logistisk regression**: `Flatten` gör om bilden till 784 tal, och `Linear(784, 10)` ger 10 logits. Det är samma modell som `LogisticRegression` i sklearn och vår baslinje inför L05.

In [ ]:
set_seed(42)
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 10)).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

t0 = time.time()
for epoch in range(3):
    train_epoch(model, train_loader, optimizer, loss_fn)
    print(f"epok {epoch+1}: val accuracy {accuracy(model, val_loader):.3f}")
print(f"tid: {time.time() - t0:.1f} s")

## 6. Vad händer om … vi inte normaliserar?

**Gissa först:** samma modell, samma learning rate och samma antal epoker, men med pixlar i 0–255, i 0–1 respektive standardiserade. Vad händer med loss och accuracy epok för epok?

In [ ]:
@torch.no_grad()
def val_loss_and_acc(model, loader):
    model.eval()
    total_loss, correct = 0.0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        out = model(X)
        total_loss += loss_fn(out, y).item() * len(y)
        correct += (out.argmax(dim=1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def run_with_transform(name, tf, lr=0.01, epochs=3):
    set_seed(42)
    full = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=tf)
    tr, va = random_split(full, [50_000, 10_000], generator=torch.Generator().manual_seed(42))
    tr_loader = DataLoader(tr, batch_size=128, shuffle=True, num_workers=2)
    va_loader = DataLoader(va, batch_size=256, num_workers=2)
    model = nn.Sequential(nn.Flatten(), nn.Linear(784, 10)).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for epoch in range(epochs):
        train_epoch(model, tr_loader, opt, loss_fn)
        v_loss, v_acc = val_loss_and_acc(model, va_loader)
        print(f"{name:16s} epok {epoch+1}: val loss {v_loss:10.3f}   val accuracy {v_acc:.3f}")

variants = {
    "0–255 (rå)": transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x * 255)]),
    "0–1 (ToTensor)": transforms.ToTensor(),
    "standardiserad": transform,
}
for name, tf in variants.items():
    run_with_transform(name, tf)

- **0–255:** gradienterna blir cirka 255 gånger större, och stegen skjuter långt över målet. Val loss hamnar i **hundratal eller tusental** i stället för runt 0,5, och den hoppar upp och ned mellan epokerna. Accuracy kan ibland se okej ut, men modellen är extremt självsäker på sina fel och träningen är instabil. Det skulle fungera med en mycket mindre learning rate, men den måste då hittas.
- **0–1:** stabil men långsammare. Indata ligger inte centrerade kring 0.
- **Standardiserad:** lägst loss och högst accuracy med samma inställningar. Det är skälet till att vi alltid normaliserar.

En lärdom på vägen: **titta på loss, inte bara accuracy.** Accuracy räknar bara rätt och fel, medan loss också visar hur säker modellen är.

## 7. Övningar

### Övning 1 – ett eget Dataset
Skriv `SquaresDataset(n)`, där exempel `i` är `(tensor([float(i)]), tensor([float(i**2)]))`.

In [ ]:
class SquaresDataset(Dataset):
    # TODO
    ...

In [ ]:
sq = SquaresDataset(100)
assert len(sq) == 100
x, y = sq[7]
assert x.item() == 7.0 and y.item() == 49.0 and x.dtype == torch.float32
xb, yb = next(iter(DataLoader(sq, batch_size=10)))
assert xb.shape == (10, 1)
print("Rätt!")

### Övning 2 – en DataLoader för en mindre datamängd
Skapa en `DataLoader` med batchstorlek 64 över de **första 5 000** exemplen i `train_ds`. Använd `torch.utils.data.Subset`. Hur många batcher blir det per epok?

In [ ]:
from torch.utils.data import Subset
small_loader = None  # TODO

In [ ]:
import math
assert len(small_loader.dataset) == 5000
assert len(small_loader) == math.ceil(5000 / 64)
print("Rätt!", len(small_loader), "batcher")

### Övning 3 – leta läckage
En kollega har skrivit koden nedan. Vad är fel, och varför spelar det roll? Skriv om den så att den blir rätt.
```python
all_pixels = raw_train.data.float() / 255     # alla 60 000 bilder
MEAN, STD = all_pixels.mean(), all_pixels.std()
...
train_ds, val_ds = random_split(full_train, [50_000, 10_000])
```
*Ditt svar:*

## När använder jag detta?
- **Eget `Dataset`** när datan inte finns i `torchvision` (egna CSV-filer, bilder i mappar, text). `torchvision.datasets.ImageFolder` täcker det vanligaste bildfallet.
- **`shuffle=True` bara för train.**
- **`num_workers` 2–4** när GPU:n väntar på data (låg GPU-användning). Använd 0 när du felsöker.
- **Normalisering**: alltid. Statistiken beräknas på train och används på val och test.
- **Förtränade modeller** (L11) kräver *samma* normalisering som de tränades med.

## Sammanfattning
- `Dataset` hämtar ett exempel och `DataLoader` gör batcher, blandar och laddar parallellt.
- Uppdelningen i train/val/test görs med `random_split` och fast seed.
- Normalisering, `(x - mean) / std`, gör att samma learning rate fungerar för alla vikter.
- Med normaliserad data gav en linjär modell cirka 83 % efter 3 epoker. Det är baslinjen inför MLP:n i L05.

**Nästa lektion:** det första neurala nätverket, en MLP med dolda lager och aktiveringsfunktioner.